In [47]:
import re
import json
import sys 
import os 
sys.path.append(os.path.abspath(".."))
from src.global_settings import CHROMA_PATH, EMBEDDING_MODEL, BM25_PATH, PROMPT_TEMPLATE, LLM_MODEL, RERANK_MODEL
import pickle
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from src.query import generate_multi_queries, dense_search, load_retrievers, hybrid_search, get_final_response, get_relevant_chunks
import cohere

In [48]:
pls_work = get_final_response('what is CUDA?')

In [28]:
response = get_final_response('what is CUDA?')

In [30]:
response.response

AttributeError: 'NoneType' object has no attribute 'response'

In [20]:
llm = ChatOpenAI(model = LLM_MODEL)

In [21]:
sample_response = llm.invoke(' just give me random 5 words')
sample_response

AIMessage(content='Sure! Here are five random words:\n\n1. Lantern  \n2. Breeze  \n3. Marble  \n4. Echo  \n5. Velvet', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 14, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5e793402c9', 'id': 'chatcmpl-DJLLob98KsCW1pC2zJ9gkazcHMUYr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cecff-3b5c-7621-be7a-1502bd064d70-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 27, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [22]:
sample_response.text

'Sure! Here are five random words:\n\n1. Lantern  \n2. Breeze  \n3. Marble  \n4. Echo  \n5. Velvet'

In [23]:
sample_response.content

'Sure! Here are five random words:\n\n1. Lantern  \n2. Breeze  \n3. Marble  \n4. Echo  \n5. Velvet'

In [9]:
co = cohere.Client(os.environ['COHERE_API_KEY'])

In [31]:
sample_query = " What is Cuda Runtime API?"

In [32]:
chroma_db, bm25, docs = load_retrievers()

In [33]:
final_docs = hybrid_search(sample_query, chroma_db, bm25, docs)

In [36]:
final_docs = get_relevant_chunks('what is CUDA?')

In [38]:
from src.query import get_response


response = get_response('what is cuda', final_docs)

In [39]:
response

AIMessage(content='I could not find an answer in the provided documentation.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 291, 'total_tokens': 302, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_e6090e88f9', 'id': 'chatcmpl-DJLSQWhFEeucVEgSWi6jjYoEKhQx8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ced05-7ec6-7fc1-be96-aaadf2bfd534-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 291, 'output_tokens': 11, 'total_tokens': 302, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [40]:
[doc.metadata.get('url', None) for doc in final_docs]

['https://docs.nvidia.com/cuda/cuda-runtime-api/index.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/annotated.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/functions.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__INTEROP.html']

In [44]:
pls_work = get_final_response('what is CUDA?')

In [42]:
pls_work

In [37]:
final_docs

[Document(id='8a286d83-376e-413d-8704-c0bd2c95f991', metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/index.html', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'chunk_id': 1780, 'h2': '', 'h3': ''}, page_content='Title : CUDA Runtime API :: CUDA Toolkit Documentation\nH2 : \nH3 :'),
 Document(id='1c5b2e76-20f8-416d-b89f-0b3d8baf04df', metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER.html', 'chunk_id': 373, 'h3': 'Functions Functions', 'h2': '6.35. Interactions with the CUDA Driver API', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation'}, page_content='on the CUDA Driver CUcontext which is current to the calling host thread. If no CUcontext is current to the calling thread when a CUDA Runtime API call which requires an active context is made, then the primary\n                        context (device execution context) for a device will be selected, made current to the calling thread, and initialized. The'),
 Docum

In [8]:
texts = [doc.page_content for doc in final_docs]

In [10]:
response = co.rerank(
    model=RERANK_MODEL,
    query=sample_query,
    documents=texts,
    top_n=5
)

In [12]:
reranked_docs = [docs[result.index] for result in response.results]

In [13]:
reranked_docs

[Document(metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/deprecated.html', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'h2': '9. Deprecated List', 'h3': '', 'chunk_id': 1}, page_content='9. Deprecated List Global cudaDeviceGetSharedMemConfig Global cudaDeviceSetSharedMemConfig Global cudaFuncSetSharedMemConfig Global cudaMemcpyArrayToArray Global cudaMemcpyFromArray Global cudaMemcpyFromArrayAsync Global cudaMemcpyToArray Global cudaMemcpyToArrayAsync Global cudaGLMapBufferObject This function is deprecated as of CUDA 3.0. Global cudaGLMapBufferObjectAsync This function is deprecated as of CUDA 3.0. Global cudaGLRegisterBufferObject This function is deprecated as of CUDA 3.0. Global cudaGLSetBufferObjectMapFlags This function is deprecated as of CUDA 3.0. Global cudaGLSetGLDevice This function is deprecated as of CUDA 5.0. Global cudaGLUnmapBufferObject This function is deprecated as of CUDA 3.0. Global cudaGLUnmapBufferObjectAsync This function is depre

In [14]:
titles = [doc.metadata.get('title', None) for doc in reranked_docs]

In [15]:
titles

['CUDA Runtime API :: CUDA Toolkit Documentation',
 'CUDA Runtime API :: CUDA Toolkit Documentation',
 'CUDA Runtime API :: CUDA Toolkit Documentation',
 'CUDA Runtime API :: CUDA Toolkit Documentation',
 'CUDA Runtime API :: CUDA Toolkit Documentation']

In [ ]:
urls = [doc.metadata.get('url', None) for doc in reranked_docs]

In [17]:
urls

['https://docs.nvidia.com/cuda/cuda-runtime-api/deprecated.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/deprecated.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/deprecated.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/deprecated.html',
 'https://docs.nvidia.com/cuda/cuda-runtime-api/functions.html']

In [18]:
[doc.metadata.get('h2', None) for doc in reranked_docs]

['9. Deprecated List',
 '9. Deprecated List',
 '9. Deprecated List',
 '9. Deprecated List',
 '8. Data Fields A B C D E F G H I K L M N O P R S T U V W X Y Z']

In [19]:
[doc.metadata.get('h3', None) for doc in reranked_docs]

['', '', '', '', '']

In [25]:
multi_queries = generate_multi_queries("How do I use CUDA streams?")

In [26]:
multi_queries.append('fddf')

In [27]:
multi_queries

['Definition and purpose of CUDA streams in GPU programming',
 'How to implement and manage concurrency with CUDA streams',
 'Best practices for optimizing performance using CUDA streams',
 'Common errors and troubleshooting when working with CUDA streams',
 'Relationship between CUDA streams and events in asynchronous execution',
 'fddf']

In [2]:
chroma_db, bm25, docs = load_retrievers()

In [3]:
sample_query = "What does the CUDA runtime function cudaGetDriverEntryPoint return?"

In [8]:
tokenized_query = re.findall(r"\w+", sample_query.lower())
scores = bm25.get_scores(tokenized_query)
top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
bm_results = [docs[i] for i in top_n]
dense_results = dense_search(chroma_db, sample_query)

In [9]:
results_list = [bm_results, dense_results]

In [35]:
results_list[0][0]

Document(metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'h2': '6.31. Driver Entry Point Access', 'h3': 'Functions Functions', 'chunk_id': 621}, page_content='6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ \u200b cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ \u200b cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __host__ \u200b cudaError_t cudaGetDriverEntryP

In [32]:
results_list[1][0]

Document(id='5f1b79f0-e22c-4e61-a91a-51ccfb6d2e29', metadata={'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'chunk_id': 621, 'h2': '6.31. Driver Entry Point Access', 'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html', 'h3': 'Functions Functions'}, page_content='6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ \u200b cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ \u200b cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __ho

In [11]:
k = 60

In [14]:
fused_scores = {}
doc_map = {}

In [15]:
for results in results_list:
    for rank, doc in enumerate(results):
        chunk_id = doc.metadata['chunk_id']
        if chunk_id not in fused_scores :
            fused_scores[chunk_id] = 0 
            doc_map[chunk_id] = doc 
        fused_scores[chunk_id] += 1 / (k+rank)


In [16]:
fused_scores

{621: 0.03333333333333333,
 283: 0.01639344262295082,
 277: 0.016129032258064516,
 538: 0.015873015873015872,
 1654: 0.015625,
 620: 0.01639344262295082,
 627: 0.016129032258064516,
 625: 0.015873015873015872,
 623: 0.015625}

In [28]:
sorted_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)

In [29]:
[doc_map[chunk_id] for chunk_id in sorted_ids]

[Document(metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'h2': '6.31. Driver Entry Point Access', 'h3': 'Functions Functions', 'chunk_id': 621}, page_content='6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ \u200b cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ \u200b cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __host__ \u200b cudaError_t cudaGetDriverEntry

In [34]:
context_text = '\n\n---\n\n '.join([doc.page_content for doc in results])
print(context_text)

6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ ​ cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ ​ cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __host__ ​ cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. Parameters symbol - The base name of the driver API function to look for. As an example, for the driver API

-

In [6]:
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt_template

ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, template='\nYou are an assistant that answers questions using NVIDIA CUDA Runtime documentation.\n\nUse ONLY the information provided in the context to answer the question.\n\nRules:\n- Do not use prior knowledge about CUDA.\n- If the answer is not present in the context, say: "I could not find an answer in the provided documentation."\n- Do not make up functions, parameters, or behavior.\n- Prefer quoting exact terminology from the documentation.\n\nContext :\n{context}\n\n---\n\nQuestion:\n{query}\n\nAnswer:\n'), additional_kwargs={})])

In [7]:
prompt = prompt_template.format(context=context_text, query=sample_query)
print(prompt)

Human: 
You are an assistant that answers questions using NVIDIA CUDA Runtime documentation.

Use ONLY the information provided in the context to answer the question.

Rules:
- Do not use prior knowledge about CUDA.
- If the answer is not present in the context, say: "I could not find an answer in the provided documentation."
- Do not make up functions, parameters, or behavior.
- Prefer quoting exact terminology from the documentation.

Context :
6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ ​ cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ ​ cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPoi

In [10]:
llm = ChatOpenAI(model=LLM_MODEL)

In [11]:
response = llm.invoke(prompt)
response

AIMessage(content='The CUDA runtime function **cudaGetDriverEntryPoint** "Returns the requested driver API function pointer."', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 881, 'total_tokens': 901, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5e793402c9', 'id': 'chatcmpl-DJJU9Ui8dBaBLTL9AuN5IPeb5Q88D', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cec91-ce67-70e0-9d58-ee58efb49b4e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 881, 'output_tokens': 20, 'total_tokens': 901, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
response.content

'The CUDA runtime function **cudaGetDriverEntryPoint** "Returns the requested driver API function pointer."'

In [ ]:
type(results[0])

In [35]:
from src.query import bm25_search


bm_results = bm25_search(bm25, docs, sample_query)

In [36]:
type(bm_results[0])

langchain_core.documents.base.Document

In [38]:
bm_results[0]

Document(metadata={'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__DRIVER__ENTRY__POINT.html', 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation', 'h2': '6.31. Driver Entry Point Access', 'h3': 'Functions Functions', 'chunk_id': 621}, page_content='6.31. Driver Entry Point Access This section describes the driver entry point access functions of CUDA runtime application programming interface. Functions __host__ \u200b cudaError_t cudaGetDriverEntryPoint (  const char* symbol , void** funcPtr , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer. __host__ \u200b cudaError_t cudaGetDriverEntryPointByVersion (  const char* symbol , void** funcPtr , unsigned int cudaVersion , unsigned long long flags , cudaDriverEntryPointQueryResult * * driverStatus = NULL ) Returns the requested driver API function pointer by CUDA version. Functions __host__ \u200b cudaError_t cudaGetDriverEntryP

In [ ]:
sample_query = "What does the CUDA runtime function cudaGetDriverEntryPoint return?"
generate_multi_queries(sample_query)

In [ ]:
db = Chroma(persist_directory=CHROMA_PATH, embedding_function=OpenAIEmbeddings(model=EMBEDDING_MODEL))
with open(BM25_PATH, "rb") as f:
    bm25, docs = pickle.load(f)

In [ ]:
print(db._collection.count())

In [ ]:
query = "What does the CUDA runtime function cudaGetDriverEntryPoint return?"

In [ ]:
vector_docs = db.similarity_search(query, k=5)

In [ ]:
vector_docs

In [ ]:
tokenized_query = query.lower().split()
scores = bm25.get_scores(tokenized_query)

In [ ]:
top_n = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
top_n

In [ ]:
bm25_docs = [docs[i] for i in top_n]
bm25_docs

In [16]:
from src.global_settings import JSON_PATH
from src.create_db import load_docs, create_chunks

docs = load_docs(JSON_PATH)

In [17]:
chunks = create_chunks(docs)

In [24]:
chunks[0].metadata

{'url': 'https://docs.nvidia.com/cuda/cuda-runtime-api/deprecated.html',
 'title': 'CUDA Runtime API :: CUDA Toolkit Documentation',
 'h2': '9. Deprecated List',
 'h3': '',
 'chunk_id': 0}

In [26]:
import re

In [27]:
re.findall(r"\w+", chunks[0].page_content.lower())

['title',
 'cuda',
 'runtime',
 'api',
 'cuda',
 'toolkit',
 'documentation',
 'h2',
 '9',
 'deprecated',
 'list',
 'h3']

In [25]:
chunks[0].page_content

'Title : CUDA Runtime API :: CUDA Toolkit Documentation\nH2 : 9. Deprecated List\nH3 :'